In [11]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("ECommerceAnalysis") \
    .getOrCreate()

# 2. Read the CSV files into DataFrames
customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)
returns_df = spark.read.csv("returns.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("order_items.csv", header=True, inferSchema=True)

# 3. Calculate and print total counts
print(f"Total Customers: {customers_df.count()}")
print(f"Total Products: {products_df.count()}")
print(f"Total Orders: {orders_df.count()}")
print(f"Total Returned Orders: {returns_df.count()}")
print(f"Total order items: {order_items_df.count()}")

customers_df.printSchema()
products_df.printSchema()
orders_df.printSchema()
returns_df.printSchema()
order_items_df.printSchema()



Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


[Stage 93:>                                                         (0 + 2) / 2]

Total order items: 3000000
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_cost: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- return_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- return_date: date (nullable = true)
 |-- return_reason: string (nullable = true)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_

In [14]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

# 2. Load the required CSV dataframes
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("order_items.csv", header=True, inferSchema=True)

# 3. Join datasets using the common 'product_id' column
joined_df = order_items_df.join(products_df, on="product_id", how="inner")

# 4. Calculate total sales per row (quantity * selling_price)
sales_df = joined_df.withColumn("sales_amount", F.col("quantity") * F.col("selling_price"))

# 5. Group by 'category', sum up sales, and convert from scientific notation to standard decimals
category_sales = sales_df.groupBy("category") \
    .agg(F.round(F.sum("sales_amount"), 2).cast("decimal(18,2)").alias("total_sales")) \
    .orderBy(F.desc("total_sales"))

# 6. Display the final clean results table
category_sales.show(truncate=False)

[Stage 113:============================>                            (1 + 1) / 2]

+--------------+------------+
|category      |total_sales |
+--------------+------------+
|Beauty        |762669305.90|
|Home & Kitchen|758138873.28|
|Books         |746490778.35|
|Toys          |744619072.30|
|Electronics   |744266504.11|
|Sports        |743338868.13|
|Clothing      |741922794.57|
+--------------+------------+



In [15]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

# 1. Load the required CSV dataframes
customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("order_items.csv", header=True, inferSchema=True)

# 2. Step-by-Step Join
# Join items to orders using 'order_id'
items_with_orders = order_items_df.join(orders_df, on="order_id", how="inner")

# Join the result to customers using 'customer_id'
full_joined_df = items_with_orders.join(customers_df, on="customer_id", how="inner")

# 3. Calculate total spending per item line (quantity * selling_price)
spending_df = full_joined_df.withColumn("item_total", F.col("quantity") * F.col("selling_price"))

# 4. Group by customer, aggregate spending, and sort to get the Top 10
top_10_customers = spending_df.groupBy("customer_id", "customer_name", "customer_segment") \
    .agg(F.round(F.sum("item_total"), 2).cast("decimal(18,2)").alias("total_purchase_amount")) \
    .orderBy(F.desc("total_purchase_amount")) \
    .limit(10)

# 5. Display the final leaderboard
top_10_customers.show(truncate=False)


[Stage 131:============================>                            (1 + 1) / 2]

+-----------+--------------+----------------+---------------------+
|customer_id|customer_name |customer_segment|total_purchase_amount|
+-----------+--------------+----------------+---------------------+
|93094      |Customer_93094|VIP             |181569.68            |
|64560      |Customer_64560|Standard        |169060.40            |
|23289      |Customer_23289|Premium         |161573.80            |
|52275      |Customer_52275|Standard        |153364.79            |
|61218      |Customer_61218|Standard        |153067.55            |
|52034      |Customer_52034|Standard        |152680.05            |
|40442      |Customer_40442|Standard        |151037.32            |
|60528      |Customer_60528|VIP             |148691.95            |
|84830      |Customer_84830|Premium         |148363.84            |
|82593      |Customer_82593|Premium         |148281.04            |
+-----------+--------------+----------------+---------------------+



In [16]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# 1. Load DataFrames
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("order_items.csv", header=True, inferSchema=True)

# 2. Join orders and order items
joined_df = order_items_df.join(orders_df, on="order_id", how="inner")

# 3. Add temporary columns for calculation, extracting Year and Month from order_date
sales_df = joined_df \
    .withColumn("sales_amount", F.col("quantity") * F.col("selling_price")) \
    .withColumn("order_year", F.year(F.col("order_date"))) \
    .withColumn("order_month", F.month(F.col("order_date")))

# 4. Find the maximum (latest) year available in your dataset dynamically
latest_year = sales_df.select(F.max("order_year")).collect()[0][0]
print(f"📈 Analyzing monthly trends for the latest available year: {latest_year}")

# 5. Filter for that year, group by month, sum sales, and sort chronologically
monthly_trends = sales_df.filter(F.col("order_year") == latest_year) \
    .groupBy("order_month") \
    .agg(F.round(F.sum("sales_amount"), 2).cast("decimal(18,2)").alias("monthly_sales")) \
    .orderBy("order_month")

# 6. Display the final trend
monthly_trends.show(truncate=False)


📈 Analyzing monthly trends for the latest available year: 2024


[Stage 149:>                                                        (0 + 2) / 2]

+-----------+-------------+
|order_month|monthly_sales|
+-----------+-------------+
|1          |444577775.76 |
|2          |415366144.20 |
|3          |443628245.41 |
|4          |427820974.34 |
|5          |444810618.95 |
|6          |431705154.06 |
|7          |443670519.12 |
|8          |441095177.02 |
|9          |431071526.08 |
|10         |441363789.31 |
|11         |433623364.04 |
|12         |442712908.35 |
+-----------+-------------+



In [17]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# 1. Load the DataFrames
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("order_items.csv", header=True, inferSchema=True)
returns_df = spark.read.csv("returns.csv", header=True, inferSchema=True)

# 2. Join Order Items with Products to get the category for every item sold
items_with_category = order_items_df.join(products_df, on="product_id", how="inner")

# 3. Left join with Returns using order_id to flag which items were sent back
# We drop duplicate order_id from returns to avoid inflating counts if joins overlap
returns_clean = returns_df.select("order_id").withColumn("is_returned", F.lit(1))
analysis_df = items_with_category.join(returns_clean, on="order_id", how="left")

# 4. Group by category and calculate total orders vs total returns
category_returns = analysis_df.groupBy("category") \
    .agg(
        F.count("order_item_id").alias("total_items_ordered"),
        F.sum(F.coalesce(F.col("is_returned"), F.lit(0))).alias("total_items_returned")
    ) \
    .withColumn(
        "return_percentage", 
        F.round((F.col("total_items_returned") / F.col("total_items_ordered")) * 100, 2)
    ) \
    .orderBy(F.desc("return_percentage"))

# 5. Display the final leaderboard
category_returns.show(truncate=False)


[Stage 162:============================>                            (1 + 1) / 2]

+--------------+-------------------+--------------------+-----------------+
|category      |total_items_ordered|total_items_returned|return_percentage|
+--------------+-------------------+--------------------+-----------------+
|Toys          |430418             |43382               |10.08            |
|Beauty        |430547             |43194               |10.03            |
|Sports        |424412             |42530               |10.02            |
|Books         |427086             |42809               |10.02            |
|Home & Kitchen|434034             |43418               |10.0             |
|Electronics   |425896             |42601               |10.0             |
|Clothing      |427607             |42660               |9.98             |
+--------------+-------------------+--------------------+-----------------+



In [33]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# 1. Load DataFrames
customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)

# 2. Join customers and orders to connect 'state' with 'payment_mode'
customer_orders = orders_df.join(customers_df, on="customer_id", how="inner")

# 3. Group by state and payment mode to get the total count for each combination
state_payment_counts = customer_orders.groupBy("state", "payment_mode") \
    .agg(F.count("order_id").alias("transaction_count"))

# 4. Define a Window to rank payment modes within each individual state
window_spec = Window.partitionBy("state").orderBy(F.desc("transaction_count"))

# 5. Apply the rank and filter for only the #1 preferred payment mode
top_payment_per_state = state_payment_counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .drop("rank") \
    .orderBy("state")

# 6. Display the final results
top_payment_per_state.show(100, truncate=False)


[Stage 213:============================>                            (1 + 1) / 2]

+-----+----------------+-----------------+
|state|payment_mode    |transaction_count|
+-----+----------------+-----------------+
|CA   |UPI             |20246            |
|FL   |Debit Card      |20010            |
|GA   |Net Banking     |20041            |
|IL   |Cash on Delivery|20498            |
|MI   |Debit Card      |20416            |
|NC   |Net Banking     |19596            |
|NY   |Debit Card      |20369            |
|OH   |Net Banking     |20351            |
|TX   |UPI             |20065            |
|WA   |UPI             |20244            |
+-----+----------------+-----------------+

